In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

##sensor_daily_features.csv 생성 (센서-일자별 데이터 요약본 생성)

# 1. 경로 설정
BASE_DIR = Path(os.getcwd()).parent
PROCESSED_DIR = BASE_DIR / "01_data" / "02_processed"
HIST_PATH = PROCESSED_DIR / "sensor_timeslot_hist.csv"
SAVE_PATH = PROCESSED_DIR / "sensor_daily_features.csv"

# 2. 데이터 로드
df = pd.read_csv(HIST_PATH)

def extract_daily_features(group):
    """하루치(6개 구간) 히스토그램을 합산하여 특징 추출"""
    db_cols = [f"db_{i}" for i in range(100)]
    daily_hist = group[db_cols].sum().values
    
    total_count = np.sum(daily_hist)
    if total_count == 0: return None
    
    nonzero_idx = np.where(daily_hist > 0)[0]
    
    # 지표 산출
    critical_value = int(nonzero_idx.min()) if len(nonzero_idx) > 0 else 0
    peak = int(np.argmax(daily_hist))
    count_peak = int(daily_hist[peak])
    peak_ratio = round((count_peak / total_count) * 100, 2)
    spread = int(nonzero_idx.max() - critical_value) if len(nonzero_idx) > 0 else 0
    
    return pd.Series({
        'critical_value': critical_value,
        'peak': peak,
        'count_peak': count_peak,
        'peak_ratio': peak_ratio,
        'spread': spread,
        'total_count': total_count
    })

# 3. 그룹화 및 저장
print("📊 일일 지표 요약 생성 중...")
daily_features = df.groupby(['date', 'site_name', 'sensor_id'], sort=False).apply(extract_daily_features).reset_index()

daily_features.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print(f"✅ 2번 파일 생성 완료: {SAVE_PATH} ({len(daily_features):,}행)")


📊 일일 지표 요약 생성 중...
✅ 2번 파일 생성 완료: c:\workspace\wvr\classification\labeling_project\01_data\02_processed\sensor_daily_features.csv (15,833행)
